# Storage v2 — paragraphs as ground truth

The new storage layer built on structure-aware splitting (`structuring.py`, the settled
recipe from `structure-split.ipynb`). Three tables in a fresh `.hrono/hrono-v2.db` — the
old `hrono.db` stays untouched as research history.

**The shape:**
- `paragraphs` — the ground truth. One row per structural paragraph (~800 B budget):
  file identity (decomposed URI parts), byte span, content hash, `struct_paths` (JSON
  list of full structure paths — one per packed node, per-segment `idx` disambiguation),
  `synthetic` flag, `chunk_no` link.
- `chunks` — same mechanism, higher budget: the SAME packer runs over the same structure
  tree at ~10 KB, so chunks get their own `struct_paths` exactly like paragraphs, and
  paragraph boundaries nest inside chunk boundaries by construction (the paragraph pack
  runs *inside* each chunk's node group). The two tables are near-identical in shape.
- `contents` — content-addressed bodies: `hash -> body`, one row per unique hash.
  Why stored in-DB at all: the git cache only holds the CURRENT checkout, while rows are
  pinned to `@version` — after a pull, old byte ranges point into different bytes, and
  non-git sources can't re-fetch old versions. The DB must be self-contained for serving.
  Chunk content is NOT duplicated — it's the concatenation of its paragraphs' bodies.
- `embeddings` (unchanged design, not rebuilt here) — keyed `(hash, model)`, works for
  paragraph vectors as-is.

**URI decision:** paragraph URIs are file-level (`git://.../File.java@version`, no
fragment) — position lives in the byte span, human context in `struct_paths`. Chunk URIs
keep `#chunk-<n>` as before.

In [1]:
import json
import sqlite3
import time
from collections import Counter
from hashlib import sha256
from pathlib import Path

from structuring import BUDGET, split_file_hier, sp_render

APP_DIR = Path.cwd()
OLD_DB = APP_DIR / ".hrono" / "hrono.db"
NEW_DB = APP_DIR / ".hrono" / "hrono-v2.db"
GIT_CACHE = APP_DIR / ".hrono" / "cache" / "git"

DDL = """
DROP TABLE IF EXISTS paragraphs;
DROP TABLE IF EXISTS chunks;
DROP TABLE IF EXISTS contents;

CREATE TABLE contents (
    hash TEXT PRIMARY KEY,   -- sha-256 of the raw bytes
    body BLOB NOT NULL       -- the raw bytes themselves; one row per unique hash (dedup)
);

CREATE TABLE chunks (
    id          INTEGER PRIMARY KEY,
    source_type TEXT    NOT NULL,
    instance    TEXT    NOT NULL,
    path        TEXT    NOT NULL,
    version     TEXT    NOT NULL,
    chunk_no    INTEGER NOT NULL,   -- ordering within the file
    byte_start  INTEGER NOT NULL,   -- = first paragraph's start
    byte_end    INTEGER NOT NULL,   -- = last paragraph's end
    hash        TEXT    NOT NULL,   -- content-address of the chunk's raw byte span
    struct_paths TEXT   NOT NULL,   -- same JSON as paragraphs: list of full paths, at chunk granularity
    struct_uris TEXT    NOT NULL,   -- struct_paths rendered as space-separated sp: URIs (Python render)
    uri TEXT GENERATED ALWAYS AS (
        source_type || '://' || instance || '/' || path || '@' || version
    ) VIRTUAL,  -- resource address (fragment-free): position is data (chunk_no/span), not identity
    UNIQUE (source_type, instance, path, version, chunk_no)
);

CREATE TABLE paragraphs (
    id          INTEGER PRIMARY KEY,
    source_type TEXT    NOT NULL,
    instance    TEXT    NOT NULL,
    path        TEXT    NOT NULL,
    version     TEXT    NOT NULL,
    in_chunk    INTEGER NOT NULL,   -- chunk_no of the chunk this paragraph is packed into
    byte_start  INTEGER NOT NULL,   -- position + ordering: no para_no needed
    byte_end    INTEGER NOT NULL,
    hash        TEXT    NOT NULL REFERENCES contents(hash),
    struct_uris TEXT    NOT NULL,   -- struct_paths rendered as space-separated sp: URIs (Python render)
    struct_paths TEXT   NOT NULL,   -- JSON list of FULL paths (one per packed node), each path
                                    -- = [{"type","name"?,"idx"}, ...]; segments typed "~..."/"raw"
                                    -- mark synthetic (delimiter-built) boundaries — no separate
                                    -- flag column: derivable data only drifts
    uri TEXT GENERATED ALWAYS AS (
        source_type || '://' || instance || '/' || path || '@' || version
    ) VIRTUAL,                       -- resource address (not unique per paragraph):
                                     -- pure query convenience, so virtual — zero bytes stored
    UNIQUE (source_type, instance, path, version, byte_start)
);
CREATE INDEX idx_paragraphs_hash ON paragraphs(hash);
CREATE INDEX idx_paragraphs_file ON paragraphs(path, byte_start);
"""

db = sqlite3.connect(NEW_DB)
db.executescript(DDL)
print(f"fresh schema in {NEW_DB.name}")

fresh schema in hrono-v2.db


## Build: split every corpus file, pack chunks, insert

File list + `@version` come from the old DB (its rows match the cache checkout).
Drop-and-rebuild per run, like the research pipeline — incremental upsert is the update
mechanism's job later; here `contents` insertion is already naturally idempotent
(`INSERT OR IGNORE` on hash).

In [2]:
old = sqlite3.connect(OLD_DB)
files = old.execute(
    "SELECT DISTINCT source_type, instance, path, version FROM chunks ORDER BY path").fetchall()
old.close()

t0 = time.monotonic()
n_paras = n_chunks = 0
for source_type, instance, path, version in files:
    fpath = GIT_CACHE / instance / path
    raw = fpath.read_bytes()
    _, file_chunks = split_file_hier(fpath)
    with db:
        for chunk_no, c in enumerate(file_chunks):
            db.execute(
                "INSERT INTO chunks (source_type, instance, path, version, chunk_no,"
                " byte_start, byte_end, hash, struct_paths, struct_uris) VALUES (?,?,?,?,?,?,?,?,?,?)",
                (source_type, instance, path, version, chunk_no, c.byte_start, c.byte_end,
                 sha256(raw[c.byte_start:c.byte_end]).hexdigest(), json.dumps(c.struct_paths),
                 " ".join(sp_render(sp) for sp in c.struct_paths)))
            n_chunks += 1
            for p in c.paragraphs:
                body = raw[p.byte_start:p.byte_end]
                h = sha256(body).hexdigest()
                db.execute("INSERT OR IGNORE INTO contents (hash, body) VALUES (?,?)", (h, body))
                db.execute(
                    "INSERT INTO paragraphs (source_type, instance, path, version, in_chunk,"
                    " byte_start, byte_end, hash, struct_paths, struct_uris) VALUES (?,?,?,?,?,?,?,?,?,?)",
                    (source_type, instance, path, version, chunk_no, p.byte_start, p.byte_end,
                     h, json.dumps(p.struct_paths), " ".join(sp_render(sp) for sp in p.struct_paths)))
                n_paras += 1

unique = db.execute("SELECT COUNT(*) FROM contents").fetchone()[0]
db_mb = NEW_DB.stat().st_size / 1e6
print(f"{len(files)} files -> {n_chunks} chunks, {n_paras} paragraphs "
      f"({unique} unique bodies — {n_paras - unique} deduplicated) "
      f"in {time.monotonic() - t0:.1f}s; {NEW_DB.name} = {db_mb:.1f} MB")

482 files -> 501 chunks, 4168 paragraphs (3898 unique bodies — 270 deduplicated) in 0.6s; hrono-v2.db = 5.9 MB


## Table previews — the tables as they are

Raw rows (first 100), no hand-picked columns beyond shortening hashes/versions for
readability. This is what a `SELECT *` essentially looks like.

In [3]:
import pandas as pd
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 200)

pd.read_sql_query("""
    SELECT id, source_type, instance, path, substr(version,1,8) AS version, in_chunk,
           byte_start, byte_end, substr(hash,1,10) AS hash, struct_paths, struct_uris, uri
    FROM paragraphs ORDER BY id LIMIT 100""", db)

,id,source_type,instance,path,version,in_chunk,byte_start,byte_end,hash,struct_paths,struct_uris,uri
0,1,git,github.com,branow/football-match-mvc/.gitattributes,69c8ad7c,0,0,54,193fbf6c44,[],,git://github.com/branow/football-match-mvc/.gitattributes@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
1,2,git,github.com,branow/football-match-mvc/.gitignore,69c8ad7c,0,0,444,795029e58e,[],,git://github.com/branow/football-match-mvc/.gitignore@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
2,3,git,github.com,branow/football-match-mvc/README.md,69c8ad7c,0,0,317,379366c859,"[[{""type"": ""section"", ""name"": ""Project Overview"", ""idx"": 0}, {""type"": ""paragraph"", ""idx"": 0}]]",sp:/Project%20Overview/.paragraph,git://github.com/branow/football-match-mvc/README.md@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
3,4,git,github.com,branow/football-match-mvc/README.md,69c8ad7c,0,317,516,3dbdb38d34,"[[{""type"": ""section"", ""name"": ""Project Overview"", ""idx"": 0}, {""type"": ""paragraph"", ""idx"": 1}]]",sp:/Project%20Overview/.paragraph[1],git://github.com/branow/football-match-mvc/README.md@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
4,5,git,github.com,branow/football-match-mvc/README.md,69c8ad7c,0,516,789,065cac96ee,"[[{""type"": ""section"", ""name"": ""Project Overview"", ""idx"": 0}, {""type"": ""section"", ""name"": ""Application Functionality"", ""idx"": 0}, {""type"": ""paragraph"", ""idx"": 0}]]",sp:/Project%20Overview/Application%20Functionality/.paragraph,git://github.com/branow/football-match-mvc/README.md@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
5,6,git,github.com,branow/football-match-mvc/README.md,69c8ad7c,0,789,1345,823f72e5d2,"[[{""type"": ""section"", ""name"": ""Project Overview"", ""idx"": 0}, {""type"": ""section"", ""name"": ""Application Functionality"", ""idx"": 0}, {""type"": ""section"", ""name"": ""Key Features"", ""idx"": 0}]]",sp:/Project%20Overview/Application%20Functionality/Key%20Features,git://github.com/branow/football-match-mvc/README.md@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
6,7,git,github.com,branow/football-match-mvc/README.md,69c8ad7c,0,1345,1671,ad4b7e11c2,"[[{""type"": ""section"", ""name"": ""Project Overview"", ""idx"": 0}, {""type"": ""section"", ""name"": ""Technical Stack"", ""idx"": 0}]]",sp:/Project%20Overview/Technical%20Stack,git://github.com/branow/football-match-mvc/README.md@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
7,8,git,github.com,branow/football-match-mvc/README.md,69c8ad7c,0,1671,1862,a59c063556,"[[{""type"": ""section"", ""name"": ""Project Overview"", ""idx"": 0}, {""type"": ""section"", ""name"": ""Database Design"", ""idx"": 0}]]",sp:/Project%20Overview/Database%20Design,git://github.com/branow/football-match-mvc/README.md@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
8,9,git,github.com,branow/football-match-mvc/README.md,69c8ad7c,0,1862,2219,709ef5f67b,"[[{""type"": ""section"", ""name"": ""Project Overview"", ""idx"": 0}, {""type"": ""section"", ""name"": ""Architecture"", ""idx"": 0}]]",sp:/Project%20Overview/Architecture,git://github.com/branow/football-match-mvc/README.md@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
9,10,git,github.com,branow/football-match-mvc/README.md,69c8ad7c,0,2219,2473,58c720d8a5,"[[{""type"": ""section"", ""name"": ""Project Overview"", ""idx"": 0}, {""type"": ""section"", ""name"": ""Data Validation"", ""idx"": 0}]]",sp:/Project%20Overview/Data%20Validation,git://github.com/branow/football-match-mvc/README.md@69c8ad7cb599b2731f289adfbdea6f8fecd3c169


In [4]:
pd.read_sql_query("""
    SELECT id, source_type, instance, path, substr(version,1,8) AS version, chunk_no,
           byte_start, byte_end, substr(hash,1,10) AS hash, struct_paths, struct_uris, uri
    FROM chunks ORDER BY id LIMIT 100""", db)

,id,source_type,instance,path,version,chunk_no,byte_start,byte_end,hash,struct_paths,struct_uris,uri
0,1,git,github.com,branow/football-match-mvc/.gitattributes,69c8ad7c,0,0,54,193fbf6c44,[],,git://github.com/branow/football-match-mvc/.gitattributes@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
1,2,git,github.com,branow/football-match-mvc/.gitignore,69c8ad7c,0,0,444,795029e58e,[],,git://github.com/branow/football-match-mvc/.gitignore@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
2,3,git,github.com,branow/football-match-mvc/README.md,69c8ad7c,0,0,3616,b65383c61b,"[[{""type"": ""section"", ""name"": ""Project Overview"", ""idx"": 0}]]",sp:/Project%20Overview,git://github.com/branow/football-match-mvc/README.md@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
3,4,git,github.com,branow/football-match-mvc/build.gradle,69c8ad7c,0,0,1359,66520df66c,[[]],sp:/,git://github.com/branow/football-match-mvc/build.gradle@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
4,5,git,github.com,branow/football-match-mvc/docker-compose.yml,69c8ad7c,0,0,620,2f0ae8ae93,[],,git://github.com/branow/football-match-mvc/docker-compose.yml@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
5,6,git,github.com,branow/football-match-mvc/gradle/wrapper/gradle-wrapper.properties,69c8ad7c,0,0,253,23b7b98fa7,[],,git://github.com/branow/football-match-mvc/gradle/wrapper/gradle-wrapper.properties@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
6,7,git,github.com,branow/football-match-mvc/gradlew,69c8ad7c,0,0,8762,a3648413b4,[[]],sp:/,git://github.com/branow/football-match-mvc/gradlew@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
7,8,git,github.com,branow/football-match-mvc/gradlew.bat,69c8ad7c,0,0,2966,57931b17dd,[[]],sp:/,git://github.com/branow/football-match-mvc/gradlew.bat@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
8,9,git,github.com,branow/football-match-mvc/settings.gradle,69c8ad7c,0,0,41,5da183f416,[],,git://github.com/branow/football-match-mvc/settings.gradle@69c8ad7cb599b2731f289adfbdea6f8fecd3c169
9,10,git,github.com,branow/football-match-mvc/src/main/java/com/footballtournaments/FootballTournamentsApplication.java,69c8ad7c,0,0,354,0c6ecaceed,[],,git://github.com/branow/football-match-mvc/src/main/java/com/footballtournaments/FootballTournamentsApplication.java@69c8ad7cb599b2731f289adfbdea6f8fecd3c169


In [5]:
rows = db.execute("SELECT hash, body FROM contents LIMIT 15").fetchall()
pd.DataFrame([{"hash": h[:10], "bytes": len(b),
               "body_preview": b.split(b"\n")[0][:70].decode("utf-8", "replace")}
              for h, b in rows])

,hash,bytes,body_preview
0,193fbf6c44,54,/gradlew text eol=lf
1,795029e58e,444,HELP.md
2,379366c859,317,# Project Overview
3,3dbdb38d34,199,"The core objective is to create a system that allows administrators, m"
4,065cac96ee,273,## Application Functionality
5,823f72e5d2,556,### Key Features
6,ad4b7e11c2,326,## Technical Stack
7,a59c063556,191,## Database Design
8,709ef5f67b,357,## Architecture
9,58c720d8a5,254,## Data Validation


## The data — paragraphs of one real file

`struct_path` rendered as a breadcrumb. Note idx disambiguation on repeated names and
the `synthetic` flag on delimiter-built boundaries.

In [6]:
def breadcrumb(struct_paths_json):
    # renders the list of full paths: one breadcrumb per packed node, |-separated
    paths = json.loads(struct_paths_json)
    if not paths or paths == [[]]:
        return "(whole file / top level)"
    rendered = []
    for segs in paths:
        parts = []
        for s in segs:
            label = s.get("name") or s["type"]
            if s.get("idx"):
                label += f"[{s['idx']}]"
            if "part" in s:
                label += f" (part {s['part']}/{s['of']})"
            parts.append(label)
        rendered.append(" / ".join(parts) or "(top)")
    return "  |  ".join(dict.fromkeys(rendered))


rows = db.execute("""
    SELECT byte_start, byte_end, struct_paths, hash FROM paragraphs
    WHERE path LIKE '%services/ContactService.java' ORDER BY byte_start""").fetchall()
print(f"ContactService.java -> {len(rows)} paragraphs\n")
for start, end, sp, h in rows:
    print(f"  [{start:>5}:{end:>5}] {breadcrumb(sp)}")

ContactService.java -> 36 paragraphs

  [    0:  118] import_declaration
  [  118:  232] import_declaration[1]  |  import_declaration[2]  |  import_declaration[3]  |  import_declaration[4]
  [  232:  363] import_declaration[5]  |  import_declaration[6]  |  import_declaration[7]
  [  363:  463] import_declaration[8]  |  import_declaration[9]
  [  463:  579] import_declaration[10]  |  import_declaration[11]
  [  579:  710] import_declaration[12]
  [  710:  815] import_declaration[14]  |  import_declaration[15]
  [  815:  917] import_declaration[16]  |  import_declaration[17]
  [  917: 1061] import_declaration[18]  |  import_declaration[19]  |  import_declaration[20]
  [ 1061: 1220] import_declaration[21]  |  import_declaration[22]
  [ 1220: 1428] ContactService / class_body / field_declaration  |  ContactService / class_body / field_declaration[1]  |  ContactService / class_body / field_declaration[2]
  [ 1428: 1792] ContactService / class_body / block_comment
  [ 1792: 2397] ContactServ

In [7]:
print("--- markdown (README.md) ---")
for start, end, sp, h in db.execute(
        """SELECT byte_start, byte_end, struct_paths, hash FROM paragraphs
           WHERE path LIKE '%webphone/README.md' ORDER BY byte_start LIMIT 12"""):
    print(f"  [{start:>5}:{end:>5}] {breadcrumb(sp)}")

print("\n--- json (uk/translation.json) ---")
for start, end, sp, h in db.execute(
        """SELECT byte_start, byte_end, struct_paths, hash FROM paragraphs
           WHERE path LIKE '%locales/uk/translation.json' ORDER BY byte_start LIMIT 12"""):
    print(f"  [{start:>5}:{end:>5}] {breadcrumb(sp)}")

--- markdown (README.md) ---
  [    0:  119] Web-Based Softphone System / paragraph
  [  119:  443] Web-Based Softphone System / 1. Project Overview / paragraph
  [  443:  568] Web-Based Softphone System / 1. Project Overview / paragraph[1]
  [  568:  979] Web-Based Softphone System / 1. Project Overview / list
  [  979: 1483] Web-Based Softphone System / 1. Project Overview / paragraph[2]
  [ 1483: 1660] Web-Based Softphone System / 1. Project Overview / paragraph[3]
  [ 1660: 1888] Web-Based Softphone System / 2. Features and Functionalities / paragraph
  [ 1888: 2097] Web-Based Softphone System / 2. Features and Functionalities / What Can You Do With It? / paragraph
  [ 2097: 2350] Web-Based Softphone System / 2. Features and Functionalities / What Can You Do With It? / list
  [ 2350: 2895] Web-Based Softphone System / 2. Features and Functionalities / What Can You Do With It? / paragraph[1]  |  Web-Based Softphone System / 2. Features and Functionalities / What Can You Do With It? 

## Working with it #1 — RAG hit rendering

What the retrieval processor injects per hit: ONE header line — URI, then each struct
path as a compact `sp:` URI (named segments percent-encoded; `.type` = unnamed segment;
`[i]` only when idx > 0; `.raw[part/of]` for chops) — then the body. Rendered by
`structuring.rag_header`, shown here for four file kinds to prove any path encodes.

In [8]:
EXAMPLES = [
    ("p.struct_uris LIKE '%getByUser%' AND p.path LIKE '%ContactService.java'", 1),
    ("p.path LIKE '%football-match-mvc/build.gradle' AND p.struct_paths LIKE '%compileOnly%'", 1),
    ("p.path LIKE '%football-match-mvc/README.md' AND p.struct_paths LIKE '%Technical Stack%'", 1),
    ("p.path LIKE '%webphone/LICENSE'", 1),
]
for where, limit in EXAMPLES:
    header, body = db.execute(f"""
        SELECT p.uri || ' ' || p.struct_uris, c.body FROM paragraphs p
        JOIN contents c ON c.hash = p.hash WHERE {where} LIMIT {limit}""").fetchone()
    print(header)
    print(body.decode("utf-8", "replace")[:120].rstrip() + " …\n")

git://github.com/branow/webphone/server/src/main/java/com/scisbo/webphone/services/ContactService.java@c6feb30927daddb0a18847819f083bf36ac30058 sp:/ContactService/.class_body/getByUser
@LogBefore("Retrieving contacts for user=#{#user}, page=#{#pageable}")
    @LogAfter("Retrieved contacts: #{#result}") …

git://github.com/branow/football-match-mvc/build.gradle@69c8ad7cb599b2731f289adfbdea6f8fecd3c169 sp:/dependencies/.block/implementation[2] sp:/dependencies/.block/compileOnly
implementation 'org.springframework.boot:spring-boot-starter-web'
    compileOnly 'org.projectlombok:lombok' …

git://github.com/branow/football-match-mvc/README.md@69c8ad7cb599b2731f289adfbdea6f8fecd3c169 sp:/Project%20Overview/Technical%20Stack
## Technical Stack

- Backend: Spring Boot MVC (Java) with JPA (Hibernate)
- Frontend: Thymeleaf, Bootstrap for UI styli …

git://github.com/branow/webphone/LICENSE@c6feb30927daddb0a18847819f083bf36ac30058 sp:/.~blank-line sp:/.~blank-line[1]
GNU AFFERO GENERAL PUBLIC LI

## Working with it #2 — chunk content = concatenation of its paragraphs

No duplicated chunk bodies: rebuild any chunk's text from `contents` and verify it
byte-identically matches the chunk's recorded span hash. Note: paragraphs skip blank-only
gaps, so reassembly stitches spans back over the original gaps — verified here on every
chunk in the DB.

In [9]:
def chunk_body(db, chunk_row_id):
    src, inst, path, ver, no, c_start, c_end = db.execute(
        "SELECT source_type, instance, path, version, chunk_no, byte_start, byte_end"
        " FROM chunks WHERE id = ?", (chunk_row_id,)).fetchone()
    raw_file = (GIT_CACHE / inst / path).read_bytes()  # only for gap bytes below
    parts, pos = [], c_start
    for start, end, h in db.execute(
            """SELECT byte_start, byte_end, hash FROM paragraphs
               WHERE path = ? AND version = ? AND in_chunk = ? ORDER BY byte_start""",
            (path, ver, no)):
        parts.append(raw_file[pos:start])  # blank gap between paragraphs (usually empty)
        parts.append(db.execute("SELECT body FROM contents WHERE hash = ?", (h,)).fetchone()[0])
        pos = end
    return b"".join(parts)


ok = 0
for row_id, want_hash in db.execute("SELECT id, hash FROM chunks"):
    assert sha256(chunk_body(db, row_id)).hexdigest() == want_hash
    ok += 1
print(f"all {ok} chunks reassemble byte-identically from paragraph contents")

all 501 chunks reassemble byte-identically from paragraph contents


## Working with it #3 — dedup in action

`contents` stores each unique body once; the same paragraph appearing in several files
(or several sources) is one blob with many references.

In [10]:
dupes = db.execute("""
    SELECT hash, COUNT(*) AS refs FROM paragraphs
    GROUP BY hash HAVING refs > 1 ORDER BY refs DESC LIMIT 3""").fetchall()
for h, refs in dupes:
    body = db.execute("SELECT body FROM contents WHERE hash = ?", (h,)).fetchone()[0]
    print("=" * 100)
    print(f"stored ONCE as {h[:16]}… ({len(body)} bytes) — referenced by {refs} paragraphs:")
    for uri, sp in db.execute(
            "SELECT uri, struct_paths FROM paragraphs WHERE hash = ? ORDER BY uri", (h,)):
        print(f"  {uri}")
        print(f"    {breadcrumb(sp)}")
    print("-" * 100)
    print(body.decode("utf-8", "replace"))
    print()

total, unique = db.execute(
    "SELECT (SELECT COUNT(*) FROM paragraphs), (SELECT COUNT(*) FROM contents)").fetchone()
print(f"{total} paragraph references -> {unique} stored bodies")
db.close()

stored ONCE as 74839aef2fcd3cc2… (117 bytes) — referenced by 9 paragraphs:
  git://github.com/branow/webphone/client/src/components/account/Accounts.tsx@c6feb30927daddb0a18847819f083bf36ac30058
    import_statement  |  import_statement[1]  |  import_statement[2]
  git://github.com/branow/webphone/client/src/components/contact/Contacts.tsx@c6feb30927daddb0a18847819f083bf36ac30058
    import_statement  |  import_statement[1]  |  import_statement[2]
  git://github.com/branow/webphone/client/src/components/contact/NumberTypeLabel.tsx@c6feb30927daddb0a18847819f083bf36ac30058
    import_statement  |  import_statement[1]  |  import_statement[2]
  git://github.com/branow/webphone/client/src/pages/account/view/AccountPageBody.tsx@c6feb30927daddb0a18847819f083bf36ac30058
    import_statement  |  import_statement[1]  |  import_statement[2]
  git://github.com/branow/webphone/client/src/pages/calls/Calls.tsx@c6feb30927daddb0a18847819f083bf36ac30058
    import_statement  |  import_statement[1]  |  i

## Notes / open items

- `embeddings` for paragraphs: same table design, embed per unique `contents.hash` with
  the experiment model (granite-97m) — not run here.
- Per-category budgets (`paragraphing.toml`) not yet wired into `structuring.py` — the
  800 B / 10 KB numbers are uniform and, like `chunk_size_kb` once was, deserve empirical
  derivation.
- `struct_path` name extraction is a per-grammar heuristic (`name`/`key` fields, heading
  text, xml tag) — unnamed grammars degrade gracefully to type+idx.
- Incremental upsert is deliberately absent (drop-and-rebuild per run); the update
  mechanism will diff on `(struct_path, hash)` pairs per file.
- Old `hrono.db`, `chunks` schema, and all v1 notebooks/scripts untouched — v2 lives in
  its own DB file until it earns the switch.